# 📊 Trabalho RNP — **Etapa 5** (versão **Kaggle**)
**UFG / INF — Análise Arquitetural do Congelamento de Camadas em Transformers Multilíngues**

> **Por que Kaggle:** o modo **Save Version → "Save & Run All (Commit)"** roda o notebook inteiro **no servidor, headless** — você pode **fechar o navegador** que ele continua. Sessão de GPU até ~9–12 h (cota ~30 h/semana). O modo rápido leva ~30–40 min.

## ⚙️ Antes de rodar (3 ajustes no painel da direita → *Settings*)
1. **Accelerator → GPU T4 x2** (ou P100).
2. **Internet → On** *(obrigatório — baixa o XLM-R e o dataset MARC; pode pedir verificação por telefone uma vez).*
3. **Add Input → Datasets** → selecione o dataset com os **5 CSVs da Etapa 1**
   (`S1_train_en_eletronicos.csv`, `S1_val_en_eletronicos.csv`, `S2_en_beleza.csv`, `S3_pt_eletronicos.csv`, `S4_pt_beleza.csv`).
   O notebook acha sozinho em `/kaggle/input/**`.

## ▶️ Como executar
- **Teste interativo:** *Run All* (vê os gráficos na hora).
- **Definitivo (fecha o navegador):** *Save Version* → **Save & Run All (Commit)**. Quando terminar, os resultados ficam na aba **Output** da versão.

## O que esta etapa faz
| Passo | Pergunta | Como |
|---|---|---|
| **A** | O *language shift* nulo (EN→PT) some também em língua **distante**? | Células **T5=JA**, **T6=ZH** e âncora **T7=EN** do MARC (domínio misto) |
| **C** | **Quais camadas** dão o efeito regularizador da C2? | Congelamentos finos **C2a→C2b→C2→C2c→C4** |
| **D** | O *early stopping* disparou? Há overfitting? | **Curvas de loss** treino × validação |

Modo rápido (1 seed, ~6 configs) para a direção; completo (3 seeds) para os números finais. Loop **resumível** e idempotente.

## Seção 0 — Setup do ambiente

In [ ]:
# 0.1 — Dependências. NO KAGGLE: ative "Internet: On" e "Accelerator: GPU T4 x2" (Settings).
# A maioria já vem na imagem do Kaggle; isto só garante as libs/versões.
!pip install -q transformers datasets evaluate accelerate scikit-learn 2>/dev/null
import transformers, datasets
print("transformers", transformers.__version__, "| datasets", datasets.__version__)
print("Se der ImportError: Settings → Internet (On) e rode esta célula de novo.")

In [ ]:
# 0.2 — Modelo + congelamento C1–C4 INLINE (self-contained: sem git clone, sem pasta src/)
import random, re, types
import numpy as np
import torch
from transformers import AutoTokenizer, XLMRobertaForSequenceClassification, set_seed

NOME_MODELO = "xlm-roberta-base"
NUM_LABELS = 2
CONFIGS = {
    "C1": {"nome": "Full Fine-Tuning", "embeddings": False, "camadas": set()},
    "C2": {"nome": "Freeze Lower",     "embeddings": True,  "camadas": set(range(0, 6))},
    "C3": {"nome": "Freeze Upper",     "embeddings": False, "camadas": set(range(6, 12))},
    "C4": {"nome": "Frozen Encoder",   "embeddings": True,  "camadas": set(range(0, 12))},
}
_PADRAO_CAMADA = re.compile(r"^roberta\.encoder\.layer\.(\d+)\.")

def fixar_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed); set_seed(seed)

def carregar_modelo(seed=42, nome_modelo=NOME_MODELO, num_labels=NUM_LABELS):
    fixar_seed(seed)
    tok = AutoTokenizer.from_pretrained(nome_modelo, use_fast=True)
    model = XLMRobertaForSequenceClassification.from_pretrained(nome_modelo, num_labels=num_labels)
    return model, tok

def contar_parametros(model):
    total = sum(p.numel() for p in model.parameters())
    treinavel = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {"total": total, "treinavel": treinavel, "congelado": total - treinavel}

def freeze_layers(model, config):
    if config not in CONFIGS:
        raise ValueError(f"config inválida: {config}")
    cfg = CONFIGS[config]
    for nome, p in model.named_parameters():
        if nome.startswith("classifier"):
            p.requires_grad = True; continue
        congelar = False
        if nome.startswith("roberta.embeddings"):
            congelar = cfg["embeddings"]
        else:
            m = _PADRAO_CAMADA.match(nome)
            if m is not None:
                congelar = int(m.group(1)) in cfg["camadas"]
        p.requires_grad = not congelar
    return contar_parametros(model)

# 'M' replica a interface do antigo src.model (o resto do notebook usa M.carregar_modelo etc.)
M = types.SimpleNamespace(CONFIGS=CONFIGS, carregar_modelo=carregar_modelo,
                          freeze_layers=freeze_layers, fixar_seed=fixar_seed,
                          contar_parametros=contar_parametros)
print("Modelo/congelamento inline OK — configs base:", list(M.CONFIGS))

In [ ]:
# 0.3 — Hardware, seeds e MODO DE EXECUÇÃO
import torch, numpy as np, pandas as pd
from IPython.display import display

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Hardware:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0),
          f"| {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("⚠️ SEM GPU — ative em Settings → Accelerator → GPU T4 x2 (ou P100).")

# ── MODO DE EXECUÇÃO ───────────────────────────────────────────────────────
# True  → 1 seed + configs essenciais (6 runs, ~30-40 min): valida o pipeline e
#         dá a DIREÇÃO dos resultados (Passo A: C1/C2 em JA/ZH; Passo C: gradiente).
# False → 3 seeds + 8 configs (24 runs, ~2-3 h): números finais, com desvio-padrão
#         e testes estatísticos para o relatório.
# Dica: rode RÁPIDO primeiro; depois mude para False e rode de novo — o loop é
#       resumível e REAPROVEITA tudo que já está no results_etapa5.csv.
MODO_RAPIDO = True
# ────────────────────────────────────────────────────────────────────────────

SEEDS = [42] if MODO_RAPIDO else [42, 123, 2024]
SEED_SPLIT = 42           # seed FIXA do split train'/val'
M.fixar_seed(SEED_SPLIT)
print(f"MODO_RAPIDO={MODO_RAPIDO} | seeds dos treinos: {SEEDS} | seed do split: {SEED_SPLIT}")

In [ ]:
# 0.4 — Armazenamento no KAGGLE: entrada via Dataset anexado; saída em /kaggle/working
import glob
from pathlib import Path

# ENTRADA: anexe (Add Input → Datasets) um dataset com os 5 CSVs da Etapa 1.
# O notebook procura os arquivos em qualquer subpasta de /kaggle/input.
ESPERADOS = ["S1_train_en_eletronicos.csv", "S1_val_en_eletronicos.csv",
             "S2_en_beleza.csv", "S3_pt_eletronicos.csv", "S4_pt_beleza.csv"]

def achar_dir_dados():
    for alvo in ESPERADOS:
        hits = glob.glob(f"/kaggle/input/**/{alvo}", recursive=True)
        if hits:
            return Path(hits[0]).parent
    return None

DIR_DATA = achar_dir_dados()
DIR_RES = Path("/kaggle/working/resultados_etapa5")
(DIR_RES / "runs").mkdir(parents=True, exist_ok=True)

if DIR_DATA is None:
    print("❌ Não achei os CSVs em /kaggle/input.")
    print("   Painel direito → Add Input → Datasets → selecione o dataset com os 5 CSVs.")
    print("   Esperados:", ESPERADOS)
else:
    faltando = [v for v in ESPERADOS if not (DIR_DATA / v).exists()]
    print("Dados :", DIR_DATA, "→", "✅ 5 CSVs presentes" if not faltando else f"❌ faltam {faltando}")
print("Saídas:", DIR_RES, "(persistem ao 'Save Version'; download na aba Output)")

### 0.5 — Configurações de congelamento estendidas (Passo C)

A função `M.freeze_layers` lê o dicionário `M.CONFIGS`, em que cada entrada tem `embeddings` (bool) e `camadas` (set de índices 0–11). Acrescentamos as novas configurações **em tempo de execução** — sem alterar o `src/model.py` versionado.

In [ ]:
# 0.5 — Estende M.CONFIGS com os congelamentos finos do Passo C
# Lembrete da hierarquia: embeddings ≈ 192M (69%) | cada camada do encoder ≈ 7M.
M.CONFIGS["C2a"] = {"nome": "Freeze Embeddings",   "embeddings": True,  "camadas": set()}
M.CONFIGS["C2b"] = {"nome": "Freeze Emb + L0-2",   "embeddings": True,  "camadas": set(range(0, 3))}
M.CONFIGS["C2c"] = {"nome": "Freeze Emb + L0-8",   "embeddings": True,  "camadas": set(range(0, 9))}
M.CONFIGS["C3b"] = {"nome": "Freeze L6-9 (só topo parcial)", "embeddings": False, "camadas": set(range(6, 10))}

# Configs a treinar — dependem do MODO_RAPIDO (definido em 0.3).
# Rápido: baseline + gradiente de congelamento (C2a→C2b→C2→C2c→C4) — cobre Passo A (C1/C2
#         em JA/ZH) e a localização do efeito (Passo C). 6 runs, a maioria "congelada" (rápida).
# Completo: acrescenta C3 (Freeze Upper) e C3b (probe L6-9) para o quadro completo.
if MODO_RAPIDO:
    CONFIGS_ETAPA5 = ["C1", "C2", "C2a", "C2b", "C2c", "C4"]
else:
    CONFIGS_ETAPA5 = ["C1", "C2", "C3", "C4", "C2a", "C2b", "C2c", "C3b"]

# Tabela de parâmetros treináveis por config (sanidade).
_m, _ = M.carregar_modelo(seed=42)
linhas = []
for cfg in CONFIGS_ETAPA5:
    c = M.freeze_layers(_m, cfg)
    linhas.append({"config": cfg, "nome": M.CONFIGS[cfg]["nome"],
                   "treináveis": c["treinavel"], "%": round(100*c["treinavel"]/c["total"], 2)})
del _m
display(pd.DataFrame(linhas))

## Seção 1 — Células de teste T1–T6

- **T1–T4** vêm da Etapa 1 (carregadas do Drive): EN/Elec, EN/Beleza, PT/Elec, PT/Beleza.
- **T5 (JA/Elec)** e **T6 (ZH/Elec)** são novas (Passo A), construídas do dataset multilíngue MARC.

O treino continua sendo **S1 = EN/Eletrônicos** (não muda). Assim, T5 e T6 medem *Language Shift* para línguas **tipologicamente distantes** do inglês.

In [ ]:
# 1.1 — Carrega T1–T4 (Etapa 1) e o conjunto de treino S1_train
ARQ = {
    "S1_train": "S1_train_en_eletronicos.csv",
    "T1":       "S1_val_en_eletronicos.csv",
    "T2":       "S2_en_beleza.csv",
    "T3":       "S3_pt_eletronicos.csv",
    "T4":       "S4_pt_beleza.csv",
}
dados = {k: pd.read_csv(DIR_DATA / v) for k, v in ARQ.items()}
for k, df in dados.items():
    print(f"{k:>9}: {len(df):>7,} linhas | colunas = {list(df.columns)}")

# N por classe das células de teste (para nivelar T5/T6 ao mesmo tamanho).
N_TEST = int((dados["T2"]["label"] == 0).sum())
print(f"\nN_test por classe (das células T1–T4): {N_TEST:,}  → {2*N_TEST:,} por célula")

### 1.2 — Construir T5 (Japonês) e T6 (Mandarim) a partir do MARC

O `amazon_reviews_multi` original foi descontinuado (fev/2024); usamos o espelho **`mteb/amazon_reviews_multi`**. A célula abaixo é **defensiva**: detecta automaticamente os nomes de coluna (variam entre espelhos) e **imprime a distribuição de categorias** para você conferir o mapeamento de domínio antes de filtrar.

In [ ]:
# 1.2a — Carrega o MARC e inspeciona o schema (JA e ZH)
from datasets import load_dataset

CANDIDATOS_REPO = ["mteb/amazon_reviews_multi", "amazon_reviews_multi", "SetFit/amazon_reviews_multi_ja"]

def carregar_marc(lang):
    """Tenta carregar o split de teste do MARC para uma língua, tolerando variações."""
    erros = []
    for repo in CANDIDATOS_REPO:
        for kwargs in ({"name": lang}, {}):  # config por língua OU split único c/ coluna language
            try:
                ds = load_dataset(repo, split="test", **kwargs)
                return ds, repo, kwargs
            except Exception as e:
                erros.append(f"{repo} {kwargs}: {str(e)[:90]}")
    raise RuntimeError("Falha ao carregar MARC:\n" + "\n".join(erros))

ds_ja_raw, repo_ja, kw_ja = carregar_marc("ja")
print(f"JA carregado de: {repo_ja} {kw_ja} | colunas = {ds_ja_raw.column_names}")
df_ja_raw = pd.DataFrame(ds_ja_raw)
display(df_ja_raw.head(3))

In [ ]:
# 1.2b — Detecta colunas. O espelho mteb/ NÃO tem categoria de produto: só texto + rótulo.
def detectar_colunas(df):
    def pick(*opts):
        for o in opts:
            if o in df.columns: return o
        return None
    return {
        "stars": pick("stars", "label", "labels", "rating"),
        "title": pick("review_title", "title"),
        "body":  pick("review_body", "text", "review", "content"),
    }

COLS = detectar_colunas(df_ja_raw)
print("Colunas detectadas:", COLS)
assert COLS["stars"] and COLS["body"], f"Schema inesperado: {df_ja_raw.columns.tolist()}"
vals = sorted(int(v) for v in pd.unique(df_ja_raw[COLS["stars"]]))
print("Valores do rótulo:", vals,
      "→", "0-4 (0=1★ … 4=5★)" if min(vals) == 0 else "1-5 (estrelas)")
print("Distribuição:", df_ja_raw[COLS["stars"]].value_counts().sort_index().to_dict())

In [ ]:
# 1.2c — Sentimento binário a partir do rótulo (auto-detecta 0-4 vs 1-5) — SEM filtro de domínio
# Este mirror do MARC não traz categoria de produto, então a célula de língua distante usa
# reviews de DOMÍNIO MISTO. Para isolar a distância linguística (sem confundir com domínio),
# a 1.2d cria também uma ÂNCORA EN do MESMO MARC (T7): assim T7→T5/T6 mede só a língua.
def balancear(S, n, seed=SEED_SPLIT):
    pos = S[S["label"] == 1]; neg = S[S["label"] == 0]
    n = min(n, len(pos), len(neg))
    if n == 0:
        return S.iloc[0:0].copy(), 0
    pos = pos.sample(n=n, random_state=seed); neg = neg.sample(n=n, random_state=seed)
    return pd.concat([pos, neg]).sample(frac=1, random_state=seed).reset_index(drop=True), n

def construir_celula_marc(df_raw, cols, idioma, n_test):
    """Binariza o sentimento (1-2★→neg, 4-5★→pos, 3★ descartada) e balanceia a n_test/classe.
    Sem filtro de categoria (o mirror não tem). Domínio = misto."""
    import numpy as np
    s = df_raw[cols["stars"]].astype(int)
    stars = s + 1 if s.min() == 0 else s                 # normaliza 0-4 → 1-5
    corpo = df_raw[cols["body"]].fillna("").astype(str)
    if cols["title"]:
        texto = (df_raw[cols["title"]].fillna("").astype(str) + ". " + corpo).str.strip()
    else:
        texto = corpo.str.strip()
    df = pd.DataFrame({
        "texto": texto.values,
        "label": stars.map(lambda n: 0 if n <= 2 else (1 if n >= 4 else np.nan)).values,
        "idioma": idioma})
    df = df[df["texto"].str.len() > 0].dropna(subset=["label"]).copy()
    df["label"] = df["label"].astype(int)
    neg = int((df["label"] == 0).sum()); pos = int((df["label"] == 1).sum())
    cel, k = balancear(df, n_test)
    print(f"  [{idioma}] disponível neg={neg:,} pos={pos:,} → célula {len(cel):,} ({k:,}/classe)")
    return cel[["idioma", "label", "texto"]]

print("Construindo T5 (JA, multi-domínio):")
T5 = construir_celula_marc(df_ja_raw, COLS, "ja", N_TEST)

In [ ]:
# 1.2d — ZH (T6) e a ÂNCORA EN do próprio MARC (T7) — mesma fonte/domínio-misto que T5/T6
ds_zh_raw, repo_zh, kw_zh = carregar_marc("zh")
df_zh_raw = pd.DataFrame(ds_zh_raw)
COLS_ZH = detectar_colunas(df_zh_raw)
print("Construindo T6 (ZH, multi-domínio):")
T6 = construir_celula_marc(df_zh_raw, COLS_ZH, "zh", N_TEST)

ds_en_raw, repo_en, kw_en = carregar_marc("en")
df_en_raw = pd.DataFrame(ds_en_raw)
COLS_EN = detectar_colunas(df_en_raw)
print("Construindo T7 (EN-MARC, âncora multi-domínio):")
T7 = construir_celula_marc(df_en_raw, COLS_EN, "en", N_TEST)

dados["T5"] = T5; dados["T6"] = T6; dados["T7"] = T7
print("\n✅ T5 (JA), T6 (ZH) e T7 (EN-MARC âncora) prontas.")

> **Nota de validade (Passo A).** O espelho `mteb/amazon_reviews_multi` **não traz categoria de produto** — só texto + nota. Por isso a língua distante é medida em **domínio misto**, e criamos a **âncora T7 = EN do próprio MARC**: como T5 (JA), T6 (ZH) e T7 (EN) compartilham fonte e composição de domínio, a comparação **T7 → T5 / T6** isola a *distância linguística* de forma limpa (o domínio é "misto" nos três). A célula T3 (PT/Eletrônicos, B2W) permanece como referência de língua próxima. Nota: o F1 absoluto em T5/T6/T7 tende a ser menor que em T1 (treino é EN/Eletrônicos; aqui o domínio é misto) — o que importa é o **degrau entre as línguas**, não o nível absoluto.

In [ ]:
# 1.3 — Split val' (10% do S1_train, só early stopping) + tokenização de tudo
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer
from datasets import Dataset

train_df, val_df = train_test_split(
    dados["S1_train"], test_size=0.10,
    stratify=dados["S1_train"]["label"], random_state=SEED_SPLIT)
print(f"train': {len(train_df):,} | val': {len(val_df):,}")

tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base", use_fast=True)
MAX_LEN = 128

def tokenizar_df(df):
    ds = Dataset.from_pandas(
        df[["texto", "label"]].rename(columns={"label": "labels"}), preserve_index=False)
    return ds.map(lambda b: tokenizer(b["texto"], truncation=True, max_length=MAX_LEN),
                  batched=True, remove_columns=["texto"])

ds_train = tokenizar_df(train_df)
ds_val   = tokenizar_df(val_df)
CELULAS = ["T1", "T2", "T3", "T4", "T5", "T6", "T7"]
ds_testes = {t: tokenizar_df(dados[t]) for t in CELULAS if len(dados[t]) > 0}
for t in CELULAS:
    n = len(dados[t]) if t in dados else 0
    print(f"  {t}: {n:,} exemplos" + ("  ⚠️ VAZIA (será ignorada)" if n == 0 else ""))
print("✅ Tokenização concluída. Células ativas:", list(ds_testes))

## Seção 2 — Treino unificado (Passos A + C) — *resumível*

Define as funções de treino **inline** (célula 2.1 — `compute_metrics`, `criar_training_args`, `get_early_stopping`), sem `%%writefile`/import, para evitar cache de módulo no Colab. Setup: AdamW, lr 2e-5, 3 épocas, *warmup* 10%, *weight decay* 0.01, fp16, *early stopping* paciência 1 em `eval_loss`; **batch 32/64 (+ `group_by_length` quando suportado)** para acelerar. O loop treina `len(CONFIGS_ETAPA5) × len(SEEDS)` modelos (6 no rápido, 24 no completo), avalia cada um em **T1–T6** e salva:
- `results_etapa5.csv` — 6 medições por modelo (T1–T6);
- `runs/run_<cfg>_<seed>.json` — histórico de loss (Passo D).

Hiperparâmetros idênticos entre configs **de propósito**: isola a variável "congelamento".

In [ ]:
# 2.1 — Funções de treino DEFINIDAS INLINE (sem %%writefile/import — evita cache de módulo)
# Motivo: editar src/train.py e reimportar NÃO recarrega de forma confiável (o pacote `src`
# guarda o atributo `train` antigo, então rodava bytecode velho). Definir aqui resolve de vez.
import numpy as np
import evaluate
from transformers import TrainingArguments, EarlyStoppingCallback

f1_metric = evaluate.load("f1")
acc_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    f1_macro = f1_metric.compute(predictions=predictions, references=labels, average="macro")["f1"]
    acc = acc_metric.compute(predictions=predictions, references=labels)["accuracy"]
    f1_classes = f1_metric.compute(predictions=predictions, references=labels, average=None)["f1"]
    return {"f1_macro": f1_macro, "accuracy": acc,
            "f1_negativo": f1_classes[0], "f1_positivo": f1_classes[1]}

def criar_training_args(out_dir, seed):
    # Aceleração vs. Etapa 3 (batch 16): batch 32/64 + group_by_length + workers (~30-40%
    # mais rápido). ROBUSTO À VERSÃO: tenta construir e, se o transformers instalado
    # rejeitar um kwarg (ex.: group_by_length removido) ou usar outro nome de estratégia,
    # remove/renomeia o argumento problemático e tenta de novo. Reage ao erro real (não
    # depende de inspect.signature, que nesta versão não bate com o runtime).
    import re
    base = dict(
        output_dir=out_dir, num_train_epochs=3, learning_rate=2e-5,
        per_device_train_batch_size=32, per_device_eval_batch_size=64,
        weight_decay=0.01, warmup_ratio=0.10, fp16=True,
        group_by_length=True, dataloader_num_workers=2,
        eval_strategy="epoch", save_strategy="epoch",
        load_best_model_at_end=True, metric_for_best_model="eval_loss",
        greater_is_better=False, seed=seed,
        logging_strategy="epoch", report_to="none")
    for _ in range(len(base) + 2):
        try:
            return TrainingArguments(**base)
        except TypeError as e:
            m = re.search(r"unexpected keyword argument '(\w+)'", str(e))
            if not m:
                raise
            bad = m.group(1)
            if bad == "eval_strategy" and "evaluation_strategy" not in base:
                base["evaluation_strategy"] = base.pop("eval_strategy")  # nome antigo
                print("  [args] 'eval_strategy' -> 'evaluation_strategy'")
            elif bad in base:
                base.pop(bad)
                print(f"  [args] kwarg não suportado removido: '{bad}'")
            else:
                raise
    return TrainingArguments(**base)

def get_early_stopping():
    return EarlyStoppingCallback(early_stopping_patience=1)

print("✅ Funções de treino prontas (inline):", compute_metrics.__name__,
      criar_training_args.__name__, get_early_stopping.__name__)

In [ ]:
# 2.2 — Loop de treino (resumível): treina, salva loss, avalia em T1–T6
# Usa as funções da célula 2.1 (inline) — NÃO importa src.train (que tinha cache teimoso).
import json, shutil, gc, torch, pandas as pd
from transformers import Trainer, DataCollatorWithPadding

RESULTS_CSV = DIR_RES / "results_etapa5.csv"

def treinar_run(config, seed):
    print(f"\n{'='*60}\n🚀 Run: Config={config} ({M.CONFIGS[config]['nome']}) | Seed={seed}\n{'='*60}")
    model, tok = M.carregar_modelo(seed)
    params = M.freeze_layers(model, config)
    print(f"[{config}-{seed}] treináveis: {params['treinavel']:,} / {params['total']:,}")

    out_dir = f"/content/tmp_{config}_{seed}"
    trainer = Trainer(
        model=model, args=criar_training_args(out_dir, seed),
        train_dataset=ds_train, eval_dataset=ds_val,
        compute_metrics=compute_metrics, callbacks=[get_early_stopping()],
        data_collator=DataCollatorWithPadding(tokenizer=tok))
    trainer.train()

    # Passo D — histórico de loss
    hist = {"config": config, "seed": seed,
            "best_eval_loss": trainer.state.best_metric,
            "log_history": trainer.state.log_history}
    with open(DIR_RES / f"runs/run_{config}_{seed}.json", "w", encoding="utf-8") as f:
        json.dump(hist, f, indent=2)

    # Passo A — avalia nas 6 células
    linhas = []
    for teste_nome, ds_teste in ds_testes.items():
        m = trainer.evaluate(eval_dataset=ds_teste, metric_key_prefix="eval")
        linhas.append({"config": config, "seed": seed, "teste": teste_nome,
                       "f1_macro": m.get("eval_f1_macro"), "accuracy": m.get("eval_accuracy"),
                       "f1_negativo": m.get("eval_f1_negativo"), "f1_positivo": m.get("eval_f1_positivo")})
    # Grava deduplicando: remove linhas antigas deste (config,seed) e reescreve o CSV inteiro
    # (arquivo pequeno). Isso atualiza runs parciais — ex.: CSV antigo sem T5/T6/T7.
    df_novo = pd.DataFrame(linhas)
    if RESULTS_CSV.exists():
        old = pd.read_csv(RESULTS_CSV)
        old = old[~((old["config"] == config) & (old["seed"] == seed))]
        pd.concat([old, df_novo], ignore_index=True).to_csv(RESULTS_CSV, index=False)
    else:
        df_novo.to_csv(RESULTS_CSV, index=False)
    print(f"[{config}-{seed}] ✅ {len(linhas)} resultados ({'/'.join(ds_testes)}) gravados.")

    if Path(out_dir).exists(): shutil.rmtree(out_dir)
    del model, trainer; gc.collect(); torch.cuda.empty_cache()

def executar_pipeline():
    # Pula um (config,seed) só se TODAS as células ativas já estão preenchidas (não-nulas).
    # Assim um CSV antigo com T5/T6 vazios força o re-treino para completá-las.
    feitos = set()
    if RESULTS_CSV.exists():
        df_old = pd.read_csv(RESULTS_CSV)
        precisa = set(ds_testes)
        for (cfg, sd), g in df_old.groupby(["config", "seed"]):
            if precisa.issubset(set(g.dropna(subset=["f1_macro"])["teste"])):
                feitos.add((cfg, int(sd)))
    for config in CONFIGS_ETAPA5:
        for seed in SEEDS:
            if (config, seed) in feitos:
                print(f"⏩ Pulando {config}/seed={seed} (completo no CSV)."); continue
            treinar_run(config, seed)
    print("\n🎉 Etapa 5 (Passos A+C) concluída.")

executar_pipeline()

## Seção 3 — Passo D: Curvas de loss

Lê os JSONs em `runs/` (re-gerados na Seção 2 — resolve o problema de "os logs ficaram no Drive da Etapa 3") e plota treino × validação por época, agregando as 3 seeds. Permite ver se o *early stopping* disparou e comparar a estabilidade entre configurações.

In [ ]:
# 3.1 — Curvas de loss (treino vs validação) por configuração
import json, glob
import matplotlib.pyplot as plt

runs = []
for p in sorted(glob.glob(str(DIR_RES / "runs/run_*.json"))):
    with open(p, encoding="utf-8") as f:
        runs.append(json.load(f))
print(f"{len(runs)} arquivos de log encontrados.")

def curvas(run):
    tr, ev = {}, {}
    for e in run["log_history"]:
        if "loss" in e and "epoch" in e:        tr[round(e["epoch"])] = e["loss"]
        if "eval_loss" in e and "epoch" in e:   ev[round(e["epoch"])] = e["eval_loss"]
    return tr, ev

cfgs = sorted({r["config"] for r in runs}, key=lambda c: CONFIGS_ETAPA5.index(c) if c in CONFIGS_ETAPA5 else 99)
ncol = 4; nrow = (len(cfgs) + ncol - 1) // ncol
fig, axes = plt.subplots(nrow, ncol, figsize=(4.2*ncol, 3.4*nrow), squeeze=False)
for ax, cfg in zip(axes.flat, cfgs):
    for r in [r for r in runs if r["config"] == cfg]:
        tr, ev = curvas(r)
        ax.plot(list(tr), list(tr.values()), "o-", alpha=.5, color="tab:blue")
        ax.plot(list(ev), list(ev.values()), "s--", alpha=.5, color="tab:orange")
    ax.set_title(f"{cfg} · {M.CONFIGS.get(cfg, {}).get('nome','')}", fontsize=9)
    ax.set_xlabel("época"); ax.set_ylabel("loss")
for ax in axes.flat[len(cfgs):]: ax.axis("off")
fig.legend(["treino", "validação"], loc="upper right")
plt.tight_layout(); plt.savefig(DIR_RES / "loss_curves.png", dpi=130, bbox_inches="tight")
plt.show()
print("Salvo:", DIR_RES / "loss_curves.png")

## Seção 4 — Análise (tabelas, Δ-shifts e testes estatísticos)

Estende a lógica da Etapa 4 para 8 configs × 6 células. Mesmos testes: **Welch t** (primário), **Mann-Whitney U** (confirmação) e **Cohen's d** (efeito), limiar p < 0.10.

Δ-shifts:
- **Domínio** = T1 − T2 (EN/Elec → EN/Beleza), baseline C1
- **Língua próxima** = T1 − T3 (EN → PT, B2W)
- **Língua distante (núcleo do Passo A)** = **T7 − T5** (EN→JA) e **T7 − T6** (EN→ZH), tudo dentro do MARC (mesma fonte/domínio-misto), isolando a língua

In [ ]:
# 4.1 — Tabela média ± desvio de F1-macro (config × célula)
df = pd.read_csv(RESULTS_CSV)
print(f"{len(df)} medições | configs: {sorted(df.config.unique())}")

media = df.pivot_table(index="config", columns="teste", values="f1_macro", aggfunc="mean")
desvio = df.pivot_table(index="config", columns="teste", values="f1_macro", aggfunc="std")
media = media.reindex([c for c in CONFIGS_ETAPA5 if c in media.index])
print("\nMédia de F1-macro:"); display(media.round(4))

import seaborn as sns, matplotlib.pyplot as plt
plt.figure(figsize=(9, 5))
sns.heatmap(media, annot=True, fmt=".3f", cmap="viridis", cbar_kws={"label": "F1-macro"})
plt.title("F1-macro médio — configuração × célula de teste (Etapa 5)")
plt.tight_layout(); plt.savefig(DIR_RES / "heatmap_etapa5.png", dpi=130); plt.show()

In [ ]:
# 4.2 — Δ-shifts por configuração (em pontos percentuais)
def delta(cfg, a, b):  # F1(a) - F1(b) em pp, média sobre seeds
    g = df[df.config == cfg].groupby("teste")["f1_macro"].mean()
    if a in g and b in g: return round(100*(g[a]-g[b]), 2)
    return float("nan")

rows = []
for cfg in media.index:
    rows.append({"config": cfg,
                 "Δ Domínio (T1-T2)":        delta(cfg, "T1", "T2"),
                 "Δ Língua próx EN-PT (T1-T3)": delta(cfg, "T1", "T3"),
                 "Δ MARC EN→JA (T7-T5)":     delta(cfg, "T7", "T5"),
                 "Δ MARC EN→ZH (T7-T6)":     delta(cfg, "T7", "T6")})
display(pd.DataFrame(rows).set_index("config"))
print("Δ > 0 = queda ao sair da referência. Passo A: as colunas MARC EN→JA/ZH são >> a EN-PT?")
print("(EN-PT usa B2W; EN→JA/ZH usam o MESMO MARC, então isolam melhor a distância de língua.)")

In [ ]:
# 4.3 — Testes estatísticos: cada config vs C1, por célula (foco T2, T5, T6)
from scipy import stats
import numpy as np

def vetor(cfg, teste):
    return df[(df.config == cfg) & (df.teste == teste)].sort_values("seed")["f1_macro"].values

def cohens_d(a, b):
    na, nb = len(a), len(b)
    sp = np.sqrt(((na-1)*np.var(a, ddof=1) + (nb-1)*np.var(b, ddof=1)) / (na+nb-2))
    return (np.mean(a)-np.mean(b)) / sp if sp > 0 else 0.0

base = "C1"
linhas = []
for teste in ["T2", "T5", "T6"]:
    b = vetor(base, teste)
    for cfg in [c for c in media.index if c != base]:
        a = vetor(cfg, teste)
        if len(a) < 2 or len(b) < 2: continue
        t_p = stats.ttest_ind(a, b, equal_var=False).pvalue
        try: u_p = stats.mannwhitneyu(a, b, alternative="two-sided").pvalue
        except ValueError: u_p = float("nan")
        linhas.append({"célula": teste, "config": cfg,
                       "ΔF1 vs C1 (pp)": round(100*(a.mean()-b.mean()), 2),
                       "Welch p": round(t_p, 3), "MWU p": round(u_p, 3),
                       "Cohen d": round(cohens_d(a, b), 2),
                       "signif (p<.10)": "✓" if t_p < 0.10 else ""})
res = pd.DataFrame(linhas)
display(res)
res.to_csv(DIR_RES / "tabela_testes_etapa5.csv", index=False)
print("Salvo:", DIR_RES / "tabela_testes_etapa5.csv")

In [ ]:
# 4.4 — Passo C: onde mora o efeito regularizador? (gradiente de congelamento em T2)
# Ordena as configs com embeddings congelados pela quantidade de camadas congeladas.
ordem_fino = ["C2a", "C2b", "C2", "C2c", "C4"]  # 0 → 3 → 6 → 9 → 12 camadas (todas c/ emb)
g_t2 = df[df.teste == "T2"].groupby("config")["f1_macro"].agg(["mean", "std"])
sub = g_t2.reindex([c for c in ordem_fino if c in g_t2.index])
print("F1-macro em T2 (Domain Shift) ao longo do gradiente de congelamento:")
display((sub*1).round(4))

import matplotlib.pyplot as plt
plt.figure(figsize=(7, 4))
plt.errorbar(range(len(sub)), sub["mean"], yerr=sub["std"], marker="o", capsize=4)
if "C1" in g_t2.index:
    plt.axhline(g_t2.loc["C1", "mean"], ls="--", color="gray", label="C1 (baseline)")
plt.xticks(range(len(sub)), sub.index)
plt.ylabel("F1-macro em T2"); plt.xlabel("← menos congelamento   ·   mais congelamento →")
plt.title("Passo C — localização do efeito de congelamento (Domain Shift)")
plt.legend(); plt.tight_layout(); plt.savefig(DIR_RES / "gradiente_freezing_T2.png", dpi=130); plt.show()

### Como ler os resultados (guia de interpretação)

- **Passo A — veredito de língua distante.** Compare, na C1, o `Δ MARC EN→JA/ZH` (T7→T5/T6) com o `Δ Língua próx EN-PT`. Se JA/ZH derrubam **muito mais** que PT, o *Language Shift* **existe** e estava oculto pela proximidade EN–PT. Se JA/ZH ficam próximos de zero, a robustez multilíngue do XLM-R se confirma como achado forte. Use T7 (EN-MARC) como referência, não T1 — T7/T5/T6 partilham fonte e domínio-misto.
- **Passo C — localização do efeito.** No gráfico 4.4, o ponto em que o F1 de T2 **para de subir** indica o conjunto mínimo de camadas responsável pelo efeito regularizador da C2. Se já a **C2a** (só embeddings) recupera o ganho, o efeito é dos *embeddings*; se exige L0–L2 (**C2b**), são as camadas léxicas iniciais.
- **Passo D — curvas.** Validação subindo enquanto treino cai = *overfitting* (early stopping deve cortar). C3 com maior dispersão entre seeds confirmaria a instabilidade observada.

In [ ]:
# 4.5 — No Kaggle os resultados ficam em /kaggle/working/resultados_etapa5/
# Persistem ao fazer "Save Version" (Commit) e podem ser baixados na aba "Output".
import glob
print("Arquivos gerados em", DIR_RES, ":")
for p in sorted(glob.glob(str(DIR_RES / "**" / "*"), recursive=True)):
    print("  ", p.replace(str(DIR_RES) + "/", ""))

## Seção 5 — Passo B (opcional): desambiguar o filtro EN por categoria

**Atenção:** este passo **troca os dados de treino** (de `amazon_polarity` filtrado por palavra-chave para MARC filtrado por categoria) e, portanto, **invalida a comparação direta** com as Etapas 1–4. Rode-o como experimento separado, salvando em `results_v2`. Objetivo: remover o confundimento "ruído de label EN (90–94%)" do veredito de *Language Shift*.

Há dois caminhos; o 5.1 (categoria nativa do MARC) é preferível por ser *ground-truth*.

In [ ]:
# 5.1 — Opção B1: reconstruir o treino EN a partir das CATEGORIAS do MARC (ground-truth)
# Pré-requisito: o split EN do MARC traz product_category (verifique como em 1.2b).
ds_en_raw, repo_en, kw_en = carregar_marc("en")
df_en_raw = pd.DataFrame(ds_en_raw)
COLS_EN = detectar_colunas(df_en_raw)
print(f"EN(MARC) de {repo_en} {kw_en} | cols: {COLS_EN}")
if COLS_EN["cat"]:
    print("\nCategorias EN (confira se electronics/beauty têm volume):")
    print(df_en_raw[COLS_EN["cat"]].astype(str).str.lower().value_counts().head(25).to_string())
else:
    print("⚠️ Sem categoria no EN — pule para a Opção B2 (classificador de domínio).")

In [ ]:
# 5.2 — Opção B1 (continuação): monta S1_train_v2 (EN/Elec por categoria) e re-treina
# Usa o split TRAIN do MARC/EN; balanceia no próprio máximo (como a Etapa 1 fazia para S1).
def construir_treino_marc_en():
    ds_tr = None
    for kwargs in ({"name": "en"}, {}):
        try: ds_tr = load_dataset(repo_en, split="train", **kwargs); break
        except Exception: pass
    assert ds_tr is not None, "Não consegui o split train do MARC/EN."
    d = pd.DataFrame(ds_tr)
    cols = detectar_colunas(d)
    df = pd.DataFrame()
    tit = d[cols["title"]].fillna("").astype(str) if cols["title"] else ""
    bod = d[cols["body"]].fillna("").astype(str) if cols["body"] else ""
    df["texto"] = (tit + ". " + bod).str.strip() if cols["title"] else bod
    df["label"] = d[cols["stars"]].apply(estrela_para_sentimento)
    df["cat"]   = d[cols["cat"]].astype(str).str.strip().str.lower()
    df = df[df["texto"].str.len() > 0].dropna(subset=["label"]); df["label"] = df["label"].astype(int)
    elec = df[df["cat"].isin(CAT_ELETRONICOS)]
    cel, n = balancear(elec, min(int((elec.label==0).sum()), int((elec.label==1).sum())))
    print(f"S1_train_v2 (EN/Elec por categoria): {len(cel):,} ({n:,}/classe)")
    return cel

S1_train_v2 = construir_treino_marc_en()
# Para rodar o experimento limpo: substitua dados["S1_train"]=S1_train_v2, refaça a
# tokenização (célula 1.3), aponte RESULTS_CSV para results_v2.csv e rode o loop 2.2.
print("➡️ Para o experimento v2: troque dados['S1_train'], re-rode 1.3 e 2.2 com RESULTS_CSV=results_v2.csv.")

In [ ]:
# 5.3 — Opção B2 (contingência): classificador de domínio zero-shot (se faltar categoria EN)
# Aplica mDeBERTa zero-shot ao texto EN com rótulos curtos e simétricos (como a auditoria da Etapa 1).
def relabel_dominio_zero_shot(textos, batch=16):
    from transformers import pipeline
    zs = pipeline("zero-shot-classification",
                  model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
                  device=0 if torch.cuda.is_available() else -1)
    LABELS = ["an electronics product", "a beauty product"]
    out = []
    for i in range(0, len(textos), batch):
        for r in zs(list(textos[i:i+batch]), LABELS, multi_label=False):
            out.append("eletronicos" if r["labels"][0] == LABELS[0] else "beleza")
    return out

print("Use relabel_dominio_zero_shot(df['texto']) para reatribuir domínio quando o EN não tiver categoria.")
print("Precisão esperada ≳ 97% (vs 90–94% do filtro por palavra-chave).")

---
## Resumo dos artefatos gerados

| Arquivo (em `MyDrive/TrabalhoRNP/resultados_etapa5/`) | Conteúdo | Passo |
|---|---|---|
| `results_etapa5.csv` | 144 medições (8 configs × 3 seeds × 6 células) | A + C |
| `runs/run_*.json` | histórico de loss dos 24 treinos | D |
| `loss_curves.png` | curvas treino × validação | D |
| `heatmap_etapa5.png` | F1-macro config × célula | A + C |
| `tabela_testes_etapa5.csv` | Welch t, MWU, Cohen's d vs C1 | A + C |
| `gradiente_freezing_T2.png` | localização do efeito (Passo C) | C |

Para o relatório final, gere `docs/RESULTADOS-Etapa5.md` com a tabela 4.1, os Δ-shifts (4.2), o veredito de língua distante (4.3) e a conclusão do Passo C (4.4).
